# DePaso · IA — Paso 1: Dataset

**Qué hace este notebook:** arma el dataset de ~1500 imágenes en 4 clases `s | m | l | xl` combinando **Open Images V7** (~70%) con **tus fotos propias** (~30%), lo limpia (dedup por hash perceptual) y lo divide en train/val/test 70/15/15.

**Dónde corre:** Colab CPU alcanza (no necesita GPU). ~15-20 min + el tiempo de cargar tus fotos.

**Salida (en tu Google Drive, `MyDrive/depaso_ml/`):** `dataset/clean/images/`, `labels.csv`, `train.csv`, `val.csv`, `test.csv`.

> ⚠️ El cuello de botella de toda la tesis son **tus fotos propias**. Empezá a sacarlas ya, en paralelo con todo lo demás. Sin ellas el dataset queda sin clase `s` real y con sesgo hacia imágenes de stock.

## 0. Setup

In [ ]:
# Montar Google Drive (acá viven el dataset y el modelo, para que no se pierdan
# cuando Colab reinicia el runtime).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clonar el repo (trae los scripts del pipeline) y fijar las rutas de trabajo.
import os
REPO_URL = 'https://github.com/martinatoffoletto/DePaso.git'
if not os.path.exists('/content/DePaso'):
    !git clone {REPO_URL} /content/DePaso
else:
    !cd /content/DePaso && git pull

ML_DIR    = '/content/DePaso/depaso_rest/ml'          # scripts del pipeline
# El dataset y el modelo viven en Drive para sobrevivir reinicios del runtime:
WORK      = '/content/drive/MyDrive/depaso_ml'
RAW_DIR   = f'{WORK}/dataset/raw'
CLEAN_DIR = f'{WORK}/dataset/clean'
MODEL_DIR = f'{WORK}/models'
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
print('ML_DIR   :', ML_DIR)
print('RAW_DIR  :', RAW_DIR)
print('CLEAN_DIR:', CLEAN_DIR)
print('MODEL_DIR:', MODEL_DIR)

In [ ]:
# Dependencias del pipeline de datos (no hace falta TensorFlow todavía).
!pip install -q fiftyone pillow imagehash pandas scikit-learn

## 1. Descargar Open Images V7 (~70% del dataset)

Baja imágenes de 4 clases y las mapea a las categorías de DePaso:

`Envelope→s` · `Box→m` · `Suitcase→l` · `Furniture→xl`.

Las condiciones de sesgo (iluminación/ángulo/fondo) quedan en `unknown` para estas —solo tus fotos propias las llevan etiquetadas a mano.

In [ ]:
# La mayoría del dataset es pública. ~300 por clase ≈ 1200 imágenes de Open Images;
# tus fotos propias son un complemento chico (~100). Subí PER_CLASS si querés más.
PER_CLASS = 300
!python {ML_DIR}/dataset/download_open_images.py --out {RAW_DIR} --per-class {PER_CLASS}

## 2. Agregar tus fotos propias (~100, opcional pero recomendado)

La **mayoría del dataset son fotos públicas** de Open Images (paso anterior). Tus fotos propias son un complemento chico (~100) que suma dos cosas concretas:

- La clase **`s`** (sobres/documentos), que Open Images casi no tiene → **poné la mayoría de tus 100 acá** (~40-50).

- Un poco de **contexto argentino** y de variedad de condiciones para el análisis de sesgos.



**No tenés que medir nada.** A cada foto le ponés la **categoría a ojo** (sobre→s, caja mediana→m, valija/caja grande→l, mueble/flete→xl). El resto de las columnas son etiquetas simples de la escena, no medidas.



Si podés, variá **iluminación** (`natural`/`artificial`/`baja`), **ángulo** (`frontal`/`cenital`/`oblicuo`) y **fondo** (`liso`/`desordenado`/`exterior`) — y sacá algunos **pares** con y sin un objeto de escala conocida (celular/botella) para el flag `has_reference_object`. Si no llegás a cubrir todo, no pasa nada: se documenta como limitación del dataset en el capítulo de sesgos.



> Cuantas menos fotos propias, más peso tienen las imágenes de stock internacionales → un poco más de *domain bias*. Es un trade-off razonable para un MVP y se declara en la tesis.



### Cómo cargarlas

1. Subí tus imágenes a `MyDrive/depaso_ml/dataset/raw/images/` (arrastrá a la carpeta en Drive, o usá el cargador de la celda de abajo).

2. Agregá una fila por foto a `raw/labels.csv` con este esquema exacto:



```

filename,category,lighting,angle,background,has_reference_object,source

mi_sobre_01.jpg,s,natural,frontal,liso,False,propia

mi_caja_03.jpg,m,baja,cenital,desordenado,True,propia

```

`category` la ponés a ojo · `has_reference_object` = `True`/`False` · `source` = `propia`. Si no sabés qué poner en lighting/angle/background, usá `unknown` — igual sirve.

In [ ]:
# (Opcional) Cargador de imágenes: subí varias y se copian a raw/images/.
from google.colab import files
import shutil, os
up = files.upload()  # abre el diálogo de selección
dst = f'{RAW_DIR}/images'
os.makedirs(dst, exist_ok=True)
for name in up:
    shutil.move(name, f'{dst}/{name}')
print(f'{len(up)} imágenes copiadas a {dst}. Ahora agregá sus filas a labels.csv.')

In [ ]:
# (Opcional) Editar labels.csv cómodo con pandas en vez de a mano.
# Descomentá y completá una fila por cada foto propia que subiste.
import pandas as pd, os
csv = f'{RAW_DIR}/labels.csv'
df = pd.read_csv(csv)
# nuevas = pd.DataFrame([
#     {'filename':'mi_caja_01.jpg','category':'m','lighting':'natural',
#      'angle':'frontal','background':'liso','has_reference_object':False,'source':'propia'},
# ])
# df = pd.concat([df, nuevas], ignore_index=True)
# df.to_csv(csv, index=False)
print('Total filas en labels.csv:', len(df))
print(df['category'].value_counts())
print('\nPropias por condición de sesgo:')
prop = df[df['source']=='propia']
for col in ['lighting','angle','background','has_reference_object']:
    print(f'  {col}:', dict(prop[col].value_counts()))

## 3. Limpiar y deduplicar

Valida el CSV, tira duplicados por hash perceptual, normaliza a RGB / lado máx 1024px / JPG calidad 90.

In [ ]:
!python {ML_DIR}/dataset/build_dataset.py --in {RAW_DIR} --out {CLEAN_DIR}

## 4. Split estratificado 70/15/15

Estratifica por categoría **y** condición de sesgo. El `test.csv` queda **congelado**: es el único insumo del análisis de sesgos, no se toca al entrenar.

In [ ]:
!python {ML_DIR}/dataset/make_splits.py --data {CLEAN_DIR}

## ✅ Listo

El dataset quedó en `MyDrive/depaso_ml/dataset/clean/`. Seguí con el notebook **`02_entrenamiento.ipynb`** (ese sí necesita GPU).